# EWC 2026 Dota 2: Dotabuff + Liquipedia -> SQLite

Этот ноутбук собирает статистику турнира **Esports World Cup 2026 по Dota 2** только из двух источников:

- Dotabuff: список матчей, длительности, победители/проигравшие, командная и индивидуальная статистика.
- Liquipedia: справочная информация о турнире, участниках, формате и итоговых местах.

OpenDota, Steam API и любые неявные источники здесь не используются. Если сайт отдаёт Cloudflare/403 на прямой запрос, положите сохранённый HTML в `cache_ewc_2026/html/` с именем, которое печатает функция `cache_name_for_url`.


In [ ]:
# Установка зависимостей. В Colab/Jupyter выполните один раз.
%pip -q install requests beautifulsoup4 pandas lxml tqdm tabulate


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import sqlite3
import time
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

LEAGUE_ID = 19785
TOURNAMENT_NAME = "Esports World Cup 2026"
DB_PATH = Path("ewc_2026.sqlite")
CACHE_DIR = Path("cache_ewc_2026")
HTML_CACHE_DIR = CACHE_DIR / "html"
JSON_CACHE_DIR = CACHE_DIR / "json"
HTML_CACHE_DIR.mkdir(parents=True, exist_ok=True)
JSON_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DOTABUFF_LEAGUE = "https://www.dotabuff.com/esports/leagues/19785-esports-world-cup-2026"
DOTABUFF_MATCHES = f"{DOTABUFF_LEAGUE}/matches"
DOTABUFF_TEAMS = f"{DOTABUFF_LEAGUE}/teams"
DOTABUFF_PLAYERS = f"{DOTABUFF_LEAGUE}/players"
DOTABUFF_DRAFTS = f"{DOTABUFF_LEAGUE}/drafts"
LIQUIPEDIA_URL = "https://liquipedia.net/dota2/Esports_World_Cup/2026"
EXPECTED_DOTABUFF_MATCH_COUNT = 159

# Прямые запросы к Dotabuff/Liquipedia могут получать Cloudflare challenge.
# Это не обход: ноутбук сначала использует локальный cache, а сеть включает только по флагу.
# По умолчанию ноутбук работает с уже готовой SQLite рядом с файлом.
# Поставьте REBUILD_FROM_SOURCES=True только если хотите заново собрать данные из Dotabuff/Liquipedia.
REBUILD_FROM_SOURCES = False
USE_NETWORK = False
POLITE_DELAY_SEC = 1.2
REQUEST_TIMEOUT = 30
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    text = value.get_text(" ", strip=True) if hasattr(value, "get_text") else str(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_int(value: Any) -> int | None:
    text = clean_text(value).replace(",", "")
    m = re.search(r"-?\d+", text)
    return int(m.group(0)) if m else None


def parse_float(value: Any) -> float | None:
    text = clean_text(value).replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(m.group(0)) if m else None


def parse_duration_to_sec(value: str | None) -> int | None:
    if not value:
        return None
    parts = [int(x) for x in str(value).strip().split(":")]
    if len(parts) == 2:
        minutes, seconds = parts
        return minutes * 60 + seconds
    if len(parts) == 3:
        hours, minutes, seconds = parts
        return hours * 3600 + minutes * 60 + seconds
    return None


def parse_match_record(value: str | None) -> tuple[int | None, int | None]:
    if not value:
        return None, None
    m = re.search(r"(\d+)\s*-\s*(\d+)", value)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)


def cache_name_for_url(url: str) -> str:
    slug = re.sub(r"^https?://", "", url).strip("/")
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", slug)
    return slug[:170] + ".html"


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        }
    )
    return s


def load_or_fetch_html(
    url: str,
    *,
    cache_name: str | None = None,
    use_network: bool = USE_NETWORK,
    force_refresh: bool = False,
    session: requests.Session | None = None,
) -> tuple[str, dict[str, Any]]:
    cache_name = cache_name or cache_name_for_url(url)
    cache_path = HTML_CACHE_DIR / cache_name
    if cache_path.exists() and not force_refresh:
        text = cache_path.read_text(encoding="utf-8")
        return text, {
            "url": url,
            "cache_file": str(cache_path),
            "status": "cache",
            "fetched_at_utc": None,
            "sha256": sha256_text(text),
        }
    if not use_network:
        raise FileNotFoundError(
            f"Нет cache-файла {cache_path}. Откройте URL в браузере, сохраните HTML сюда и повторите запуск: {url}"
        )
    session = session or make_session()
    resp = session.get(url, timeout=REQUEST_TIMEOUT)
    if resp.status_code in {403, 429}:
        raise RuntimeError(
            f"{url} вернул {resp.status_code}. Сохраните страницу браузером в {cache_path} и запустите снова."
        )
    resp.raise_for_status()
    text = resp.text
    cache_path.write_text(text, encoding="utf-8")
    time.sleep(POLITE_DELAY_SEC)
    return text, {
        "url": url,
        "cache_file": str(cache_path),
        "status": f"http_{resp.status_code}",
        "fetched_at_utc": utc_now(),
        "sha256": sha256_text(text),
    }


## Парсинг Dotabuff

Парсеры не завязаны на один CSS-класс. Они ищут таблицы и ссылки вида `/matches/<id>`, `/esports/series/<id>`, `/esports/teams/...`, `/players/...`.


In [ ]:
MATCH_HREF_RE = re.compile(r"(?:https?://(?:www\.)?dotabuff\.com)?/matches/(\d+)")
SERIES_HREF_RE = re.compile(r"(?:https?://(?:www\.)?dotabuff\.com)?/esports/series/(\d+)")
TEAM_HREF_RE = re.compile(r"(?:https?://(?:www\.)?dotabuff\.com)?/esports/teams/")
PLAYER_HREF_RE = re.compile(r"(?:https?://(?:www\.)?dotabuff\.com)?/players/(\d+)")


def team_name_from_cell(cell) -> str | None:
    if cell is None:
        return None
    link = cell.find("a", href=TEAM_HREF_RE)
    if link:
        text = clean_text(link)
        if text and not text.lower().startswith("image"):
            return text
    img = cell.find("img", alt=True)
    if img and img["alt"]:
        return re.sub(r"^Image:\s*", "", img["alt"]).strip()
    text = clean_text(cell)
    text = re.sub(r"Image:\s*", "", text).strip()
    return text or None


def parse_dotabuff_matches_page(html: str, page_url: str) -> list[dict[str, Any]]:
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict[str, Any]] = []
    for tr in soup.find_all("tr"):
        match_link = tr.find("a", href=lambda href: bool(href and MATCH_HREF_RE.search(href)))
        if not match_link:
            continue
        match = MATCH_HREF_RE.search(match_link.get("href", ""))
        if not match:
            continue
        cells = tr.find_all(["td", "th"])
        row_text = clean_text(tr)
        date_match = re.search(r"20\d{2}-\d{2}-\d{2}", row_text)
        durations = re.findall(r"\b(?:\d+:)?\d{1,2}:\d{2}\b", row_text)
        series_link = tr.find("a", href=SERIES_HREF_RE)
        series_id = None
        if series_link:
            sm = SERIES_HREF_RE.search(series_link.get("href", ""))
            series_id = int(sm.group(1)) if sm else None
        winner_name = team_name_from_cell(cells[2]) if len(cells) > 2 else None
        loser_name = team_name_from_cell(cells[3]) if len(cells) > 3 else None
        duration = durations[-1] if durations else None
        match_id = int(match.group(1))
        rows.append(
            {
                "match_id": match_id,
                "league_id": LEAGUE_ID,
                "series_id": series_id,
                "match_date": date_match.group(0) if date_match else None,
                "duration_sec": parse_duration_to_sec(duration),
                "winner_name": winner_name,
                "loser_name": loser_name,
                "source_dotabuff_url": f"https://www.dotabuff.com/matches/{match_id}",
                "index_page_url": page_url,
            }
        )
    return rows


def dotabuff_match_page_diagnostics(html: str, page_url: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    title = clean_text(soup.find("title")) or clean_text(soup.find("h1")) or "(без title/h1)"
    hrefs = [
        a.get("href", "")
        for a in soup.find_all("a", href=True)
        if MATCH_HREF_RE.search(a.get("href", ""))
    ]
    sample = ", ".join(hrefs[:5]) if hrefs else "нет ссылок на /matches/<id>"
    text_head = clean_text(soup)[:300]
    return (
        f"URL: {page_url}\n"
        f"title/h1: {title}\n"
        f"найдено match-ссылок: {len(hrefs)}; пример: {sample}\n"
        f"начало текста страницы: {text_head}"
    )


def collect_dotabuff_match_index(max_pages: int = 20, use_network: bool = USE_NETWORK) -> pd.DataFrame:
    session = make_session()
    seen: set[int] = set()
    rows: list[dict[str, Any]] = []
    for page in range(1, max_pages + 1):
        url = DOTABUFF_MATCHES if page == 1 else f"{DOTABUFF_MATCHES}?page={page}"
        html, _source = load_or_fetch_html(url, use_network=use_network, session=session)
        page_rows = parse_dotabuff_matches_page(html, url)
        if page == 1 and not page_rows:
            raise ValueError(
                "Не удалось извлечь матчи из HTML-кэша Dotabuff. "
                "Чаще всего это значит, что сохранён не HTML страницы Matches, "
                "а Cloudflare challenge/overview/пустая страница.\n\n"
                + dotabuff_match_page_diagnostics(html, url)
            )
        new_rows = [r for r in page_rows if r["match_id"] not in seen]
        if not new_rows:
            break
        for r in new_rows:
            seen.add(r["match_id"])
        rows.extend(new_rows)
    expected_cols = [
        "match_id",
        "league_id",
        "series_id",
        "match_date",
        "duration_sec",
        "winner_name",
        "loser_name",
        "source_dotabuff_url",
        "index_page_url",
    ]
    df = pd.DataFrame(rows, columns=expected_cols)
    if df.empty:
        raise ValueError("Не собрано ни одного матча Dotabuff; см. диагностику выше.")
    df = df.sort_values(["match_date", "match_id"], ascending=[False, False])
    if len(df) != EXPECTED_DOTABUFF_MATCH_COUNT:
        print(
            f"WARNING: Dotabuff сообщает {EXPECTED_DOTABUFF_MATCH_COUNT} матчей, "
            f"а собрано {len(df)}. Проверьте пагинацию/cache."
        )
    return df


In [ ]:
def first_number(text: str) -> int | None:
    m = re.search(r"\d+", clean_text(text).replace(",", ""))
    return int(m.group(0)) if m else None


def parse_dotabuff_team_stats(html: str) -> list[dict[str, Any]]:
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict[str, Any]] = []
    for tr in soup.find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) < 8:
            continue
        team = team_name_from_cell(cells[0])
        if not team:
            continue
        values = [clean_text(c) for c in cells]
        match_wins, match_losses = parse_match_record(values[2] if len(values) > 2 else "")
        rows.append(
            {
                "team_name": team,
                "series_record": values[1] if len(values) > 1 else None,
                "match_record": values[2] if len(values) > 2 else None,
                "match_wins": match_wins,
                "match_losses": match_losses,
                "heroes_contested": first_number(values[3]) if len(values) > 3 else None,
                "kda": parse_float(values[4]) if len(values) > 4 else None,
                "kills_avg": parse_float(values[5]) if len(values) > 5 else None,
                "deaths_avg": parse_float(values[6]) if len(values) > 6 else None,
                "assists_avg": parse_float(values[7]) if len(values) > 7 else None,
                "last_hits_avg": parse_float(values[8]) if len(values) > 8 else None,
                "denies_avg": parse_float(values[9]) if len(values) > 9 else None,
                "gold_per_min": parse_int(values[-2]) if len(values) >= 2 else None,
                "xp_per_min": parse_int(values[-1]) if len(values) >= 1 else None,
                "source_url": DOTABUFF_TEAMS,
            }
        )
    return rows


def parse_dotabuff_player_stats(html: str) -> list[dict[str, Any]]:
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict[str, Any]] = []
    for tr in soup.find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) < 8:
            continue
        player_link = cells[0].find("a", href=PLAYER_HREF_RE)
        if not player_link:
            continue
        player_name = clean_text(player_link)
        team_name = clean_text(cells[0]).replace(player_name, "", 1).strip() or None
        href = player_link.get("href", "")
        account_id_match = PLAYER_HREF_RE.search(href)
        account_id = int(account_id_match.group(1)) if account_id_match else None
        values = [clean_text(c) for c in cells]
        match_wins, match_losses = parse_match_record(values[1] if len(values) > 1 else "")
        rows.append(
            {
                "account_id": account_id,
                "player_name": player_name,
                "team_name": team_name,
                "match_record": values[1] if len(values) > 1 else None,
                "match_wins": match_wins,
                "match_losses": match_losses,
                "kda": parse_float(values[2]) if len(values) > 2 else None,
                "kills_avg": parse_float(values[3]) if len(values) > 3 else None,
                "deaths_avg": parse_float(values[4]) if len(values) > 4 else None,
                "assists_avg": parse_float(values[5]) if len(values) > 5 else None,
                "last_hits_avg": parse_float(values[6]) if len(values) > 6 else None,
                "denies_avg": parse_float(values[7]) if len(values) > 7 else None,
                "gold_per_min": parse_int(values[-2]) if len(values) >= 2 else None,
                "xp_per_min": parse_int(values[-1]) if len(values) >= 1 else None,
                "source_url": DOTABUFF_PLAYERS,
            }
        )
    return rows


## Liquipedia

Liquipedia используется для справочной турнирной информации. Если HTML-таблицы поменялись, ноутбук сохранит сырые таблицы в `liquipedia_raw_tables`, чтобы агент мог ссылаться на источник без выдумывания фактов.


In [ ]:
TEAM_ALIASES = {
    "BB Team": "BoomBoys",
    "Poor Rangers": "_PowerRangers",
    "PowerRangers": "_PowerRangers",
    "Level UP": "Level UP esports",
    "IC x Insanity": "Inner Circle x Insanity",
    "PARIVISION": "PVISION",
    "PARI": "PVISION",
    "PV": "PVISION",
    "VG": "Vici Gaming",
    "RE": "Rune Eaters",
    "TSpirit": "Team Spirit",
    "FLCN": "Team Falcons",
    "Liquid": "Team Liquid",
    "NGX": "Nigma Galaxy",
}


# Fallback, если автоматический парсер Liquipedia не смог уверенно найти таблицу итоговых мест.
# Значения нужно считать черновыми, пока `source_url` не проверен вручную.
LIQUIPEDIA_STANDINGS_FALLBACK = [
    ("PVISION", "1", 1),
    ("BoomBoys", "2", 1),
    ("Team Yandex", "3", 1),
    ("Vici Gaming", "4", 1),
    ("Nigma Galaxy", "5-8", 1),
    ("Rune Eaters", "5-8", 0),
    ("Team Falcons", "5-8", 1),
    ("Team Spirit", "5-8", 1),
    ("1w", "9-12", 1),
    ("Aurora Gaming", "9-12", 1),
    ("LGD Gaming", "9-12", 1),
    ("Team Liquid", "9-12", 1),
    ("MOUZ", "13-16", 0),
    ("PTime", "13-16", 0),
    ("Virtus.pro", "13-16", 0),
    ("Xtreme Gaming", "13-16", 1),
    ("GamerLegion", "17-20", 1),
    ("Level UP esports", "17-20", 0),
    ("OG", "17-20", 1),
    ("REKONIX", "17-20", 0),
    ("Inner Circle x Insanity", "21-24", 0),
    ("L1 TEAM", "21-24", 1),
    ("_PowerRangers", "21-24", 0),
    ("Team Nemesis", "21-24", 0),
]


def canonical_team_name(name: str | None) -> str | None:
    if name is None:
        return None
    name = clean_text(name)
    return TEAM_ALIASES.get(name, name)


def parse_liquipedia_tables(html: str) -> tuple[dict[str, str], list[dict[str, Any]], list[dict[str, Any]]]:
    soup = BeautifulSoup(html, "html.parser")
    info: dict[str, str] = {
        "source_url": LIQUIPEDIA_URL,
        "page_title": clean_text(soup.find("h1")) or TOURNAMENT_NAME,
    }
    text = clean_text(soup)
    for key, pattern in {
        "start_date": r"Start Date:\s*(20\d{2}-\d{2}-\d{2})",
        "end_date": r"End Date:\s*(20\d{2}-\d{2}-\d{2})",
        "teams": r"Teams:\s*(\d+)",
        "patch": r"Version:\s*([0-9.]+[a-z]?)",
    }.items():
        m = re.search(pattern, text)
        if m:
            info[key] = m.group(1)

    raw_tables: list[dict[str, Any]] = []
    standings: list[dict[str, Any]] = []
    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        tables = []
    for i, df in enumerate(tables):
        df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
        raw_tables.append(
            {
                "table_index": i,
                "columns_json": json.dumps([str(c) for c in df.columns], ensure_ascii=False),
                "rows_json": df.to_json(orient="records", force_ascii=False),
                "source_url": LIQUIPEDIA_URL,
            }
        )
        cols = " ".join(str(c).lower() for c in df.columns)
        if any(token in cols for token in ["place", "placement", "приз", "prize"]):
            # Liquipedia часто меняет имена колонок; этот блок намеренно консервативен.
            pass
    if not standings:
        standings = [
            {
                "team_name": team,
                "canonical_team_name": canonical_team_name(team),
                "placement": placement,
                "ti_qualified": ti,
                "source_url": LIQUIPEDIA_URL,
                "parse_status": "fallback_needs_manual_review",
            }
            for team, placement, ti in LIQUIPEDIA_STANDINGS_FALLBACK
        ]
    return info, standings, raw_tables


## SQLite schema


In [ ]:
SCHEMA_SQL = '''
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS metadata (
    key TEXT PRIMARY KEY,
    value TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS source_pages (
    source_name TEXT NOT NULL,
    url TEXT NOT NULL,
    fetched_at_utc TEXT,
    status TEXT NOT NULL,
    cache_file TEXT,
    sha256 TEXT,
    notes TEXT,
    PRIMARY KEY (source_name, url)
);

CREATE TABLE IF NOT EXISTS team_aliases (
    alias TEXT PRIMARY KEY,
    canonical_team_name TEXT NOT NULL,
    source_name TEXT NOT NULL,
    source_url TEXT
);

CREATE TABLE IF NOT EXISTS matches (
    match_id INTEGER PRIMARY KEY,
    league_id INTEGER NOT NULL,
    series_id INTEGER,
    match_date TEXT,
    duration_sec INTEGER,
    winner_name TEXT,
    loser_name TEXT,
    radiant_name TEXT,
    dire_name TEXT,
    radiant_score INTEGER,
    dire_score INTEGER,
    radiant_win INTEGER CHECK (radiant_win IN (0,1)),
    source_dotabuff_url TEXT NOT NULL,
    fetched_at_utc TEXT,
    quality_status TEXT NOT NULL DEFAULT 'index_only'
);

CREATE TABLE IF NOT EXISTS team_match_stats (
    match_id INTEGER NOT NULL,
    team_name TEXT NOT NULL,
    opponent_name TEXT,
    side TEXT CHECK(side IN ('radiant','dire','unknown')),
    won INTEGER CHECK(won IN (0,1)),
    kills INTEGER,
    deaths INTEGER,
    gold_per_min INTEGER,
    duration_sec INTEGER,
    source_dotabuff_url TEXT,
    PRIMARY KEY (match_id, team_name),
    FOREIGN KEY (match_id) REFERENCES matches(match_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS player_match_stats (
    match_id INTEGER NOT NULL,
    team_name TEXT,
    side TEXT CHECK(side IN ('radiant','dire','unknown')),
    account_id INTEGER,
    player_name TEXT NOT NULL,
    hero_name TEXT,
    hero_id INTEGER,
    position INTEGER,
    kills INTEGER,
    deaths INTEGER,
    assists INTEGER,
    gold_per_min INTEGER,
    xp_per_min INTEGER,
    last_hits INTEGER,
    denies INTEGER,
    net_worth INTEGER,
    hero_damage INTEGER,
    tower_damage INTEGER,
    hero_healing INTEGER,
    source_dotabuff_url TEXT,
    FOREIGN KEY (match_id) REFERENCES matches(match_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS dotabuff_team_stats (
    team_name TEXT PRIMARY KEY,
    canonical_team_name TEXT,
    series_record TEXT,
    match_record TEXT,
    match_wins INTEGER,
    match_losses INTEGER,
    heroes_contested INTEGER,
    kda REAL,
    kills_avg REAL,
    deaths_avg REAL,
    assists_avg REAL,
    last_hits_avg REAL,
    denies_avg REAL,
    gold_per_min INTEGER,
    xp_per_min INTEGER,
    source_url TEXT
);

CREATE TABLE IF NOT EXISTS dotabuff_player_stats (
    account_id INTEGER,
    player_name TEXT NOT NULL,
    team_name TEXT,
    canonical_team_name TEXT,
    match_record TEXT,
    match_wins INTEGER,
    match_losses INTEGER,
    kda REAL,
    kills_avg REAL,
    deaths_avg REAL,
    assists_avg REAL,
    last_hits_avg REAL,
    denies_avg REAL,
    gold_per_min INTEGER,
    xp_per_min INTEGER,
    source_url TEXT,
    PRIMARY KEY (player_name, team_name)
);

CREATE TABLE IF NOT EXISTS tournament_info (
    key TEXT PRIMARY KEY,
    value TEXT,
    source_url TEXT
);

CREATE TABLE IF NOT EXISTS tournament_standings (
    team_name TEXT PRIMARY KEY,
    canonical_team_name TEXT NOT NULL,
    placement TEXT,
    ti_qualified INTEGER,
    source_url TEXT,
    parse_status TEXT
);

CREATE TABLE IF NOT EXISTS liquipedia_raw_tables (
    table_index INTEGER PRIMARY KEY,
    columns_json TEXT NOT NULL,
    rows_json TEXT NOT NULL,
    source_url TEXT NOT NULL
);
'''

VIEWS_SQL = '''
DROP VIEW IF EXISTS v_team_match_stats;
CREATE VIEW v_team_match_stats AS
SELECT
    t.*,
    COALESCE(a.canonical_team_name, t.team_name) AS canonical_team_name,
    COALESCE(oa.canonical_team_name, t.opponent_name) AS canonical_opponent_name
FROM team_match_stats t
LEFT JOIN team_aliases a ON a.alias = t.team_name
LEFT JOIN team_aliases oa ON oa.alias = t.opponent_name;

DROP VIEW IF EXISTS team_summary;
CREATE VIEW team_summary AS
SELECT
    canonical_team_name AS team_name,
    COUNT(*) AS games,
    SUM(CASE WHEN won = 1 THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN won = 0 THEN 1 ELSE 0 END) AS losses,
    ROUND(1.0 * SUM(CASE WHEN won = 1 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 4) AS win_rate,
    SUM(kills) AS total_kills,
    ROUND(AVG(kills), 2) AS avg_kills,
    MAX(kills) AS max_kills,
    SUM(deaths) AS total_deaths,
    ROUND(AVG(deaths), 2) AS avg_deaths,
    ROUND(AVG(duration_sec), 2) AS avg_duration_sec,
    MAX(duration_sec) AS max_duration_sec,
    ROUND(AVG(gold_per_min), 2) AS avg_gold_per_min
FROM v_team_match_stats
GROUP BY canonical_team_name;

DROP VIEW IF EXISTS team_top5_longest;
CREATE VIEW team_top5_longest AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY canonical_team_name
            ORDER BY duration_sec DESC, match_id
        ) AS rn
    FROM v_team_match_stats
    WHERE duration_sec IS NOT NULL
)
SELECT * FROM ranked WHERE rn <= 5;

DROP VIEW IF EXISTS player_summary;
CREATE VIEW player_summary AS
SELECT
    player_name,
    COALESCE(a.canonical_team_name, p.team_name) AS team_name,
    COUNT(*) AS games,
    SUM(kills) AS total_kills,
    ROUND(AVG(kills), 2) AS avg_kills,
    ROUND(AVG(deaths), 2) AS avg_deaths,
    ROUND(AVG(assists), 2) AS avg_assists,
    ROUND((SUM(kills) + SUM(assists)) * 1.0 / NULLIF(SUM(deaths), 0), 2) AS kda,
    ROUND(AVG(gold_per_min), 2) AS avg_gold_per_min,
    ROUND(AVG(xp_per_min), 2) AS avg_xp_per_min,
    ROUND(AVG(last_hits), 2) AS avg_last_hits,
    ROUND(AVG(denies), 2) AS avg_denies
FROM player_match_stats p
LEFT JOIN team_aliases a ON a.alias = p.team_name
WHERE player_name IS NOT NULL
GROUP BY player_name, COALESCE(a.canonical_team_name, p.team_name);

DROP VIEW IF EXISTS match_evidence;
CREATE VIEW match_evidence AS
SELECT
    m.match_id,
    m.match_date,
    ROUND(m.duration_sec / 60.0, 2) AS duration_min,
    m.winner_name,
    m.loser_name,
    m.radiant_name,
    m.dire_name,
    m.radiant_score,
    m.dire_score,
    m.source_dotabuff_url,
    m.quality_status
FROM matches m;

DROP VIEW IF EXISTS team_standings_joined;
CREATE VIEW team_standings_joined AS
SELECT
    s.canonical_team_name AS team_name,
    s.placement,
    s.ti_qualified,
    ts.games,
    ts.wins,
    ts.losses,
    ts.total_kills,
    ts.avg_duration_sec,
    s.source_url
FROM tournament_standings s
LEFT JOIN team_summary ts ON ts.team_name = s.canonical_team_name;

DROP VIEW IF EXISTS db_health;
CREATE VIEW db_health AS
SELECT 'matches' AS metric, CAST(COUNT(*) AS TEXT) AS value FROM matches
UNION ALL SELECT 'team_rows', CAST(COUNT(*) AS TEXT) FROM team_match_stats
UNION ALL SELECT 'player_rows', CAST(COUNT(*) AS TEXT) FROM player_match_stats
UNION ALL SELECT 'teams', CAST(COUNT(DISTINCT canonical_team_name) AS TEXT) FROM v_team_match_stats
UNION ALL SELECT 'expected_dotabuff_match_count', value FROM metadata WHERE key = 'expected_dotabuff_match_count'
UNION ALL SELECT 'build_status', value FROM metadata WHERE key = 'build_status';
'''


def init_db(path: Path = DB_PATH, overwrite: bool = True) -> sqlite3.Connection:
    if overwrite and path.exists():
        path.unlink()
    conn = sqlite3.connect(path)
    conn.executescript(SCHEMA_SQL)
    conn.executescript(VIEWS_SQL)
    return conn


In [ ]:
def upsert_metadata(conn: sqlite3.Connection, **items: Any) -> None:
    conn.executemany(
        "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
        [(str(k), str(v)) for k, v in items.items()],
    )


def upsert_source_page(conn: sqlite3.Connection, source_name: str, source: dict[str, Any], notes: str = "") -> None:
    conn.execute(
        """
        INSERT OR REPLACE INTO source_pages
        (source_name, url, fetched_at_utc, status, cache_file, sha256, notes)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (
            source_name,
            source.get("url"),
            source.get("fetched_at_utc"),
            source.get("status", "unknown"),
            source.get("cache_file"),
            source.get("sha256"),
            notes,
        ),
    )


def seed_aliases(conn: sqlite3.Connection, names: list[str] | None = None) -> None:
    alias_rows = []
    for alias, canonical in TEAM_ALIASES.items():
        alias_rows.append((alias, canonical, "manual_alias", LIQUIPEDIA_URL))
    for name in names or []:
        if name:
            alias_rows.append((name, canonical_team_name(name), "observed_name", None))
    conn.executemany(
        """
        INSERT OR REPLACE INTO team_aliases(alias, canonical_team_name, source_name, source_url)
        VALUES (?, ?, ?, ?)
        """,
        alias_rows,
    )


def upsert_matches_from_index(conn: sqlite3.Connection, df: pd.DataFrame) -> None:
    for _, row in df.iterrows():
        conn.execute(
            """
            INSERT OR REPLACE INTO matches
            (match_id, league_id, series_id, match_date, duration_sec, winner_name, loser_name,
             radiant_name, dire_name, radiant_score, dire_score, radiant_win,
             source_dotabuff_url, fetched_at_utc, quality_status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                int(row["match_id"]),
                LEAGUE_ID,
                row.get("series_id"),
                row.get("match_date"),
                row.get("duration_sec"),
                canonical_team_name(row.get("winner_name")),
                canonical_team_name(row.get("loser_name")),
                None,
                None,
                None,
                None,
                None,
                row["source_dotabuff_url"],
                utc_now(),
                "index_only",
            ),
        )
        for team, opponent, won in [
            (row.get("winner_name"), row.get("loser_name"), 1),
            (row.get("loser_name"), row.get("winner_name"), 0),
        ]:
            if team:
                conn.execute(
                    """
                    INSERT OR REPLACE INTO team_match_stats
                    (match_id, team_name, opponent_name, side, won, kills, deaths, gold_per_min, duration_sec, source_dotabuff_url)
                    VALUES (?, ?, ?, 'unknown', ?, NULL, NULL, NULL, ?, ?)
                    """,
                    (
                        int(row["match_id"]),
                        canonical_team_name(team),
                        canonical_team_name(opponent),
                        won,
                        row.get("duration_sec"),
                        row["source_dotabuff_url"],
                    ),
                )


def upsert_dotabuff_team_stats(conn: sqlite3.Connection, rows: list[dict[str, Any]]) -> None:
    for r in rows:
        conn.execute(
            """
            INSERT OR REPLACE INTO dotabuff_team_stats
            (team_name, canonical_team_name, series_record, match_record, match_wins, match_losses,
             heroes_contested, kda, kills_avg, deaths_avg, assists_avg, last_hits_avg, denies_avg,
             gold_per_min, xp_per_min, source_url)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                r.get("team_name"),
                canonical_team_name(r.get("team_name")),
                r.get("series_record"),
                r.get("match_record"),
                r.get("match_wins"),
                r.get("match_losses"),
                r.get("heroes_contested"),
                r.get("kda"),
                r.get("kills_avg"),
                r.get("deaths_avg"),
                r.get("assists_avg"),
                r.get("last_hits_avg"),
                r.get("denies_avg"),
                r.get("gold_per_min"),
                r.get("xp_per_min"),
                r.get("source_url"),
            ),
        )


def upsert_dotabuff_player_stats(conn: sqlite3.Connection, rows: list[dict[str, Any]]) -> None:
    for r in rows:
        conn.execute(
            """
            INSERT OR REPLACE INTO dotabuff_player_stats
            (account_id, player_name, team_name, canonical_team_name, match_record, match_wins, match_losses,
             kda, kills_avg, deaths_avg, assists_avg, last_hits_avg, denies_avg, gold_per_min, xp_per_min, source_url)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                r.get("account_id"),
                r.get("player_name"),
                r.get("team_name"),
                canonical_team_name(r.get("team_name")),
                r.get("match_record"),
                r.get("match_wins"),
                r.get("match_losses"),
                r.get("kda"),
                r.get("kills_avg"),
                r.get("deaths_avg"),
                r.get("assists_avg"),
                r.get("last_hits_avg"),
                r.get("denies_avg"),
                r.get("gold_per_min"),
                r.get("xp_per_min"),
                r.get("source_url"),
            ),
        )


def upsert_liquipedia(conn: sqlite3.Connection, info: dict[str, str], standings: list[dict[str, Any]], raw_tables: list[dict[str, Any]]) -> None:
    conn.executemany(
        "INSERT OR REPLACE INTO tournament_info(key, value, source_url) VALUES (?, ?, ?)",
        [(k, str(v), info.get("source_url", LIQUIPEDIA_URL)) for k, v in info.items()],
    )
    conn.executemany(
        """
        INSERT OR REPLACE INTO tournament_standings
        (team_name, canonical_team_name, placement, ti_qualified, source_url, parse_status)
        VALUES (:team_name, :canonical_team_name, :placement, :ti_qualified, :source_url, :parse_status)
        """,
        standings,
    )
    conn.executemany(
        """
        INSERT OR REPLACE INTO liquipedia_raw_tables(table_index, columns_json, rows_json, source_url)
        VALUES (:table_index, :columns_json, :rows_json, :source_url)
        """,
        raw_tables,
    )


## Открыть готовую базу или пересобрать источники


In [ ]:
# Единая настройка пути к актуальной компактной базе
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/fantasy-analytics")]
DATABASE_CANDIDATES = []
for root in PROJECT_CANDIDATES:
    DATABASE_CANDIDATES.extend([
        root / "data" / "ewc_2026_fantasy_compact.sqlite",
        root / "data" / "db" / "ewc_2026_fantasy_compact.sqlite",
        root / "ewc_2026_fantasy_compact.sqlite",
        root / "ewc_2026.sqlite",
    ])
seen = set()
DATABASE_CANDIDATES = [p for p in DATABASE_CANDIDATES if not (str(p) in seen or seen.add(str(p)))]

DB_PATH = next((path for path in DATABASE_CANDIDATES if path.exists()), DATABASE_CANDIDATES[0])

if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(VIEWS_SQL)
    conn.commit()
    print(f"Используется база данных: {DB_PATH.resolve()}")
else:
    print(f"ВНИМАНИЕ: файл базы не найден: {DB_PATH}")
    print("Положите ewc_2026_fantasy_compact.sqlite в data/, data/db/, рядом с ноутбуком или в /content/.")
    conn = init_db(DB_PATH, overwrite=True)


def show_query(title: str, sql: str, limit: int = 25, connection: sqlite3.Connection = conn) -> pd.DataFrame:
    """Выполнить SELECT-запрос и показать первые строки."""
    print(f"\n== {title} ==")
    df = pd.read_sql_query(sql, connection)
    display(df.head(limit))
    return df


def sql_list(values: list[str]) -> str:
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


display(pd.read_sql_query("SELECT * FROM db_health", conn))
display(pd.read_sql_query("SELECT * FROM team_summary ORDER BY wins DESC, total_kills DESC LIMIT 10", conn))


## Анализ игроков: официальные имена, позиции и fantasy-очки

В компактной базе `position` у игроков часто пустой, а `player_name` иногда содержит текущий внутриигровой ник. На страницах отдельных матчей Dotabuff это обычно можно проверить точнее, но в текущем локальном cache есть не HTML этих страниц, а cached match JSON. Поэтому ниже используется такой порядок:

1. `name` из cached match JSON как официальный pro-handle.
2. Ручная правка по `account_id` для известных игроков/позиций.
3. Match-level восстановление позиции по фарму внутри команды как fallback.

- `player_official_profiles_temp`: официальное имя и позиция, где они известны; иначе позиция восстановлена по среднему фарму внутри команды.
- `player_game_fantasy_analysis`: fantasy по каждой карте с `official_player_name`, `source_personaname`, `position_used` и источниками имени/позиции.
- `fantasy_team_map_role_analysis`: fantasy-композиция по карте: среднее core `(1 и 3)`, mid `(2)`, среднее support `(4 и 5)`.

Формула композиции команды на карту:

```text
team_role_fantasy_score = avg(core pos1+pos3) + mid(pos2) + avg(support pos4+pos5)
```


In [ ]:
import sqlite3
import pandas as pd
import json
from pathlib import Path

# Ручные исправления используются только для позиции, если она известна заранее.
# Имена ниже обычно подтягиваются автоматически из OpenDota match JSON: `name` = pro-handle,
# `personaname` = текущий публичный/игровой ник.
OFFICIAL_PLAYER_OVERRIDES = {
    1044002267: ("Satanic", 1),
    106573901: ("No[o]ne-", 2),
    195108598: ("DM", 3),
    164199202: ("9Class", 4),
    73401082: ("Dukalis", 5),

    100058342: ("skiter", 1),
    183719386: ("Malr1ne", 2),
    898455820: ("ATF", 3),
    25907144: ("Cr1t-", 4),
    10366616: ("Sneyking", 5),

    97590558: ("miCKe", 1),
    201358612: ("Nisha", 2),
    152962063: ("SabeRLight-", 3),
    77490514: ("Boxi", 4),
    16497807: ("tOfu", 5),

    321580662: ("Yatoro", 1),
    106305042: ("Larl", 2),
    302214028: ("Collapse", 3),
    218231587: ("rue", 4),
    847565596: ("Miposhka", 5),

    898754153: ("Ame", 1),
    173978074: ("NothingToSay", 2),
    129958758: ("Xxs", 3),
    101695162: ("XinQ", 4),
    94296097: ("xNova", 5),
}


def find_opendota_fantasy_cache() -> Path | None:
    candidates = [
        DB_PATH.parent / "cache_ewc_2026" / "opendota_fantasy",
        Path(r"D:\test\test\cache_ewc_2026\opendota_fantasy"),
        Path("/content/cache_ewc_2026/opendota_fantasy"),
        Path("cache_ewc_2026/opendota_fantasy"),
    ]
    return next((path for path in candidates if path.exists()), None)


def load_opendota_identity_rows() -> list[tuple]:
    cache_dir = find_opendota_fantasy_cache()
    if cache_dir is None:
        return []

    rows: list[tuple] = []
    for json_path in sorted(cache_dir.glob("*.json")):
        try:
            match_id = int(json_path.stem)
            data = json.loads(json_path.read_text(encoding="utf-8"))
        except Exception:
            continue

        for player in data.get("players", []):
            player_slot = player.get("player_slot")
            side = "radiant" if isinstance(player_slot, int) and player_slot < 128 else "dire"
            rows.append(
                (
                    match_id,
                    side,
                    player.get("account_id"),
                    player.get("name"),
                    player.get("personaname"),
                    player_slot,
                    player.get("lane_role"),
                    1 if player.get("is_roaming") else 0,
                )
            )
    return rows


def ensure_player_analysis_views(connection: sqlite3.Connection) -> None:
    connection.execute("DROP TABLE IF EXISTS temp.player_name_overrides_temp")
    connection.execute(
        """
        CREATE TEMP TABLE player_name_overrides_temp (
            account_id INTEGER PRIMARY KEY,
            official_player_name TEXT NOT NULL,
            official_position INTEGER NOT NULL
        )
        """
    )
    connection.executemany(
        """
        INSERT INTO player_name_overrides_temp(account_id, official_player_name, official_position)
        VALUES (?, ?, ?)
        """,
        [(account_id, name, position) for account_id, (name, position) in OFFICIAL_PLAYER_OVERRIDES.items()],
    )

    connection.execute("DROP TABLE IF EXISTS temp.player_match_identity_temp")
    connection.execute(
        """
        CREATE TEMP TABLE player_match_identity_temp (
            match_id INTEGER,
            side TEXT,
            account_id INTEGER,
            opendota_name TEXT,
            opendota_personaname TEXT,
            player_slot INTEGER,
            lane_role INTEGER,
            is_roaming INTEGER,
            PRIMARY KEY (match_id, account_id)
        )
        """
    )
    identity_rows = load_opendota_identity_rows()
    if identity_rows:
        connection.executemany(
            """
            INSERT OR REPLACE INTO player_match_identity_temp
            (match_id, side, account_id, opendota_name, opendota_personaname,
             player_slot, lane_role, is_roaming)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            identity_rows,
        )

    for view_name in [
        "player_official_profiles_temp",
        "player_map_position_temp",
        "player_game_fantasy_analysis",
        "fantasy_team_map_role_analysis",
        "fantasy_player_role_totals",
    ]:
        connection.execute(f"DROP VIEW IF EXISTS temp.{view_name}")

    connection.executescript(
        """
        CREATE TEMP VIEW player_official_profiles_temp AS
        WITH player_name_base AS (
            SELECT
                p.account_id,
                p.team_name,
                p.player_name AS raw_player_name,
                MAX(NULLIF(TRIM(i.opendota_name), '')) AS opendota_name,
                MAX(NULLIF(TRIM(i.opendota_personaname), '')) AS opendota_personaname
            FROM player_match_stats p
            LEFT JOIN player_match_identity_temp i
              ON i.match_id = p.match_id
             AND i.account_id = p.account_id
            GROUP BY p.account_id, p.team_name, p.player_name
        ),
        player_base AS (
            SELECT
                p.account_id,
                p.team_name,
                p.player_name AS raw_player_name,
                COUNT(*) AS maps_played,
                ROUND(AVG(COALESCE(p.last_hits, 0)), 2) AS avg_last_hits,
                ROUND(AVG(COALESCE(p.gold_per_min, 0)), 2) AS avg_gpm,
                ROUND(AVG(COALESCE(p.kills, 0)), 2) AS avg_kills,
                ROUND(AVG(COALESCE(p.assists, 0)), 2) AS avg_assists,
                ROW_NUMBER() OVER (
                    PARTITION BY p.team_name
                    ORDER BY
                        AVG(COALESCE(p.last_hits, 0)) DESC,
                        AVG(COALESCE(p.gold_per_min, 0)) DESC,
                        AVG(COALESCE(p.kills, 0)) DESC
                ) AS farm_rank
            FROM player_match_stats p
            GROUP BY p.account_id, p.team_name, p.player_name
        )
        SELECT
            b.account_id,
            b.team_name,
            b.raw_player_name,
            COALESCE(o.official_player_name, n.opendota_name, b.raw_player_name) AS official_player_name,
            n.opendota_personaname AS source_personaname,
            COALESCE(o.official_position, b.farm_rank) AS position_used,
            CASE COALESCE(o.official_position, b.farm_rank)
                WHEN 1 THEN 'carry'
                WHEN 2 THEN 'mid'
                WHEN 3 THEN 'offlane'
                WHEN 4 THEN 'soft_support'
                WHEN 5 THEN 'hard_support'
                ELSE 'unknown'
            END AS official_role,
            CASE
                WHEN COALESCE(o.official_position, b.farm_rank) IN (1, 3) THEN 'core'
                WHEN COALESCE(o.official_position, b.farm_rank) = 2 THEN 'mid'
                WHEN COALESCE(o.official_position, b.farm_rank) IN (4, 5) THEN 'support'
                ELSE 'unknown'
            END AS fantasy_role_group,
            CASE
                WHEN o.account_id IS NOT NULL THEN 'manual_account_id_override'
                ELSE 'inferred_by_team_avg_last_hits_rank'
            END AS position_source,
            CASE
                WHEN o.official_player_name IS NOT NULL THEN 'manual_account_id_override'
                WHEN n.opendota_name IS NOT NULL THEN 'opendota_match_json_name'
                ELSE 'player_match_stats_player_name'
            END AS name_source,
            b.maps_played,
            b.avg_last_hits,
            b.avg_gpm,
            b.avg_kills,
            b.avg_assists
        FROM player_base b
        LEFT JOIN player_name_base n
          ON n.account_id = b.account_id
         AND n.team_name = b.team_name
         AND n.raw_player_name = b.raw_player_name
        LEFT JOIN player_name_overrides_temp o
          ON o.account_id = b.account_id;

        CREATE TEMP VIEW player_map_position_temp AS
        WITH base AS (
            SELECT
                pgfs.*,
                i.opendota_name,
                i.opendota_personaname,
                i.player_slot,
                i.lane_role,
                i.is_roaming,
                ROW_NUMBER() OVER (
                    PARTITION BY pgfs.match_id, pgfs.team_name
                    ORDER BY
                        COALESCE(pgfs.last_hits, 0) DESC,
                        COALESCE(pgfs.gpm, 0) DESC,
                        COALESCE(pgfs.kills, 0) DESC,
                        COALESCE(pgfs.assists, 0) DESC
                ) AS match_farm_rank
            FROM player_game_fantasy_summary pgfs
            LEFT JOIN player_match_identity_temp i
              ON i.match_id = pgfs.match_id
             AND i.account_id = pgfs.account_id
        )
        SELECT
            b.*,
            COALESCE(o.official_player_name, b.opendota_name, b.player_name) AS official_player_name,
            COALESCE(b.opendota_personaname, b.player_name) AS source_personaname,
            COALESCE(o.official_position, b.match_farm_rank) AS position_used,
            CASE COALESCE(o.official_position, b.match_farm_rank)
                WHEN 1 THEN 'carry'
                WHEN 2 THEN 'mid'
                WHEN 3 THEN 'offlane'
                WHEN 4 THEN 'soft_support'
                WHEN 5 THEN 'hard_support'
                ELSE 'unknown'
            END AS official_role,
            CASE
                WHEN COALESCE(o.official_position, b.match_farm_rank) IN (1, 3) THEN 'core'
                WHEN COALESCE(o.official_position, b.match_farm_rank) = 2 THEN 'mid'
                WHEN COALESCE(o.official_position, b.match_farm_rank) IN (4, 5) THEN 'support'
                ELSE 'unknown'
            END AS fantasy_role_group,
            CASE
                WHEN o.official_position IS NOT NULL THEN 'manual_account_id_override'
                WHEN b.opendota_name IS NOT NULL THEN 'opendota_match_json_name_plus_match_farm_rank'
                ELSE 'match_farm_rank'
            END AS position_source,
            CASE
                WHEN o.official_player_name IS NOT NULL THEN 'manual_account_id_override'
                WHEN b.opendota_name IS NOT NULL THEN 'opendota_match_json_name'
                ELSE 'player_game_fantasy_summary_player_name'
            END AS name_source
        FROM base b
        LEFT JOIN player_name_overrides_temp o
          ON o.account_id = b.account_id;

        CREATE TEMP VIEW player_game_fantasy_analysis AS
        SELECT
            pgfs.match_id,
            m.match_date,
            pgfs.team_name,
            pgfs.opponent_name,
            pgfs.official_player_name,
            pgfs.player_name AS raw_player_name,
            pgfs.source_personaname,
            pgfs.account_id,
            pgfs.hero_name,
            pgfs.position_used,
            pgfs.official_role,
            pgfs.fantasy_role_group,
            pgfs.position_source,
            pgfs.name_source,
            pgfs.lane_role,
            pgfs.is_roaming,
            pgfs.kills,
            pgfs.deaths,
            pgfs.assists,
            pgfs.last_hits,
            pgfs.denies,
            pgfs.gpm,
            pgfs.xpm,
            ROUND(m.duration_sec / 60.0, 2) AS duration_min,
            pgfs.player_map_fantasy_score AS stored_fantasy_score,
            ROUND(
                CASE
                    WHEN pgfs.position_used IN (1, 3) THEN
                        COALESCE(pgfs.kills_points, 0) * 2.50
                      + COALESCE(pgfs.creep_score_points, 0) * 2.50
                      + COALESCE(pgfs.teamfight_participation_points, 0) * 1.80
                    WHEN pgfs.position_used = 2 THEN
                        COALESCE(pgfs.creep_score_points, 0) * 2.70
                      + COALESCE(pgfs.runes_grabbed_points, 0) * 1.80
                      + COALESCE(pgfs.teamfight_participation_points, 0) * 2.70
                    WHEN pgfs.position_used IN (4, 5) THEN
                        COALESCE(pgfs.lotus_points, 0) * 3.20
                      + COALESCE(pgfs.watchers_taken_points, 0) * 2.10
                      + COALESCE(pgfs.teamfight_participation_points, 0) * 1.50
                    ELSE pgfs.player_map_fantasy_score
                END,
                2
            ) AS fantasy_points_by_official_position,
            pgfs.score_from_kills,
            pgfs.score_from_creep_score,
            pgfs.score_from_runes,
            pgfs.score_from_watchers,
            pgfs.score_from_lotuses,
            pgfs.score_from_teamfight,
            m.source_dotabuff_url
        FROM player_map_position_temp pgfs
        JOIN matches m ON m.match_id = pgfs.match_id
        ;

        CREATE TEMP VIEW fantasy_team_map_role_analysis AS
        SELECT
            match_id,
            match_date,
            team_name,
            opponent_name,
            ROUND(AVG(CASE WHEN position_used IN (1, 3) THEN fantasy_points_by_official_position END), 2) AS avg_core_points_pos1_pos3,
            ROUND(MAX(CASE WHEN position_used = 2 THEN fantasy_points_by_official_position END), 2) AS mid_points_pos2,
            ROUND(AVG(CASE WHEN position_used IN (4, 5) THEN fantasy_points_by_official_position END), 2) AS avg_support_points_pos4_pos5,
            ROUND(
                COALESCE(AVG(CASE WHEN position_used IN (1, 3) THEN fantasy_points_by_official_position END), 0)
              + COALESCE(MAX(CASE WHEN position_used = 2 THEN fantasy_points_by_official_position END), 0)
              + COALESCE(AVG(CASE WHEN position_used IN (4, 5) THEN fantasy_points_by_official_position END), 0),
                2
            ) AS team_role_fantasy_score,
            COUNT(*) AS players_in_map
        FROM player_game_fantasy_analysis
        GROUP BY match_id, match_date, team_name, opponent_name;

        CREATE TEMP VIEW fantasy_player_role_totals AS
        SELECT
            official_player_name,
            raw_player_name,
            team_name,
            position_used,
            official_role,
            fantasy_role_group,
            position_source,
            name_source,
            COUNT(*) AS maps_played,
            ROUND(SUM(fantasy_points_by_official_position), 2) AS total_map_points,
            ROUND(AVG(fantasy_points_by_official_position), 2) AS avg_map_points,
            ROUND(MAX(fantasy_points_by_official_position), 2) AS best_map_points,
            ROUND(SUM(stored_fantasy_score), 2) AS stored_total_map_points
        FROM player_game_fantasy_analysis
        GROUP BY
            official_player_name,
            raw_player_name,
            team_name,
            position_used,
            official_role,
            fantasy_role_group,
            position_source,
            name_source;
        """
    )
    connection.commit()


ensure_player_analysis_views(conn)

top_teams_list = pd.read_sql_query(
    """
    SELECT DISTINCT canonical_team_name
    FROM tournament_standings
    WHERE placement IN ('1', '2', '3', '4', '5-8', '9-12', '13-16')
    """,
    conn,
)["canonical_team_name"].dropna().tolist()

top_16_team_names_sql = sql_list(top_teams_list)

print(f"Команд в TOP-16 фильтре: {len(top_teams_list)}")
identity_rows_loaded = conn.execute("SELECT COUNT(*) FROM player_match_identity_temp").fetchone()[0]
print(f"OpenDota identity rows loaded: {identity_rows_loaded}")
print("Временные view для анализа игроков созданы.")

show_query(
    "Проверка официальных имён и позиций",
    """
    SELECT
      team_name,
      raw_player_name,
      official_player_name,
      position_used,
      official_role,
      fantasy_role_group,
      position_source,
      name_source,
      maps_played,
      avg_last_hits,
      avg_gpm
    FROM player_official_profiles_temp
    ORDER BY team_name, position_used
    """,
    limit=30,
)


## Валидация и быстрые аналитические проверки


In [ ]:
checks = {
    "top_avg_duration": f"""
        SELECT team_name, games, ROUND(avg_duration_sec / 60.0, 2) AS avg_duration_min, max_duration_sec
        FROM team_summary
        WHERE team_name IN ({top_16_team_names_sql})
        ORDER BY avg_duration_sec DESC
        LIMIT 10
    """,
    "top_total_kills": f"""
        SELECT team_name, games, total_kills, avg_kills, max_kills
        FROM team_summary
        WHERE team_name IN ({top_16_team_names_sql})
        ORDER BY total_kills DESC
        LIMIT 10
    """,
    "top5_longest_sum": f"""
        WITH sums AS (
            SELECT canonical_team_name AS team_name, SUM(duration_sec) AS top5_total_sec, COUNT(*) AS n
            FROM team_top5_longest
            WHERE canonical_team_name IN ({top_16_team_names_sql})
            GROUP BY canonical_team_name
            HAVING COUNT(*) = 5
        )
        SELECT team_name, ROUND(top5_total_sec / 60.0, 2) AS top5_total_min
        FROM sums
        ORDER BY top5_total_sec DESC
        LIMIT 10
    """,
    "players_by_kda": f"""
        SELECT player_name, team_name, games, kda, avg_kills, avg_deaths, avg_assists, avg_gold_per_min
        FROM player_summary
        WHERE team_name IN ({top_16_team_names_sql})
        ORDER BY kda DESC
        LIMIT 10
    """,
}

for name, sql in checks.items():
    show_query(name, sql)


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

show_query(
    "Полная информация по одной карте игрока из TOP-16",
    f"""
    SELECT
        a.*,
        m.winner_name AS match_winner,
        m.loser_name AS match_loser,
        m.radiant_name,
        m.dire_name,
        m.radiant_score,
        m.dire_score
    FROM player_game_fantasy_analysis a
    JOIN matches m USING(match_id)
    WHERE a.team_name IN ({top_16_team_names_sql})
      AND a.opponent_name IN ({top_16_team_names_sql})
    ORDER BY a.match_date DESC, a.match_id DESC, a.team_name, a.position_used
    LIMIT 1
    """,
    limit=1,
)

pd.reset_option("display.max_columns")
pd.reset_option("display.width")


In [ ]:
show_query(
    "Схема основной fantasy-таблицы",
    "PRAGMA table_info(player_game_fantasy_summary);",
    limit=100,
)


In [ ]:
show_query(
    "Схема аналитической view с официальными именами",
    "PRAGMA table_info(player_game_fantasy_analysis);",
    limit=100,
)


In [ ]:
# TOP-16 команды берутся из tournament_standings по местам 1-16.
display(pd.DataFrame({"top_16_team": top_teams_list}))


In [ ]:
show_query(
    "Топ игроков по сумме fantasy-очков на независимых картах",
    f"""
    SELECT
      official_player_name,
      raw_player_name,
      team_name,
      position_used,
      official_role,
      maps_played,
      total_map_points,
      avg_map_points,
      best_map_points,
      stored_total_map_points,
      position_source
    FROM fantasy_player_role_totals
    WHERE team_name IN ({top_16_team_names_sql})
    ORDER BY total_map_points DESC
    LIMIT 25
    """,
)


In [ ]:
show_query(
    "Топ-25 fantasy-карт среди TOP-16 команд",
    f"""
    SELECT
      match_id,
      match_date,
      team_name,
      opponent_name,
      official_player_name,
      raw_player_name,
      hero_name,
      position_used,
      official_role,
      ROUND(duration_min, 2) AS duration_min,
      ROUND(fantasy_points_by_official_position, 2) AS fantasy_points,
      ROUND(stored_fantasy_score, 2) AS old_stored_fantasy_points,
      position_source
    FROM player_game_fantasy_analysis
    WHERE team_name IN ({top_16_team_names_sql})
      AND opponent_name IN ({top_16_team_names_sql})
    ORDER BY fantasy_points_by_official_position DESC
    LIMIT 25
    """,
)


In [ ]:
show_query(
    "Fantasy-анализ команды по карте: avg core + mid + avg support",
    f"""
    SELECT
      match_id,
      match_date,
      team_name,
      opponent_name,
      avg_core_points_pos1_pos3,
      mid_points_pos2,
      avg_support_points_pos4_pos5,
      team_role_fantasy_score,
      players_in_map
    FROM fantasy_team_map_role_analysis
    WHERE team_name IN ({top_16_team_names_sql})
      AND opponent_name IN ({top_16_team_names_sql})
    ORDER BY team_role_fantasy_score DESC
    LIMIT 25
    """,
)

show_query(
    "Средний fantasy-профиль команды по ролям за турнир",
    f"""
    SELECT
      team_name,
      ROUND(AVG(avg_core_points_pos1_pos3), 2) AS avg_core_pair_points,
      ROUND(AVG(mid_points_pos2), 2) AS avg_mid_points,
      ROUND(AVG(avg_support_points_pos4_pos5), 2) AS avg_support_pair_points,
      ROUND(AVG(team_role_fantasy_score), 2) AS avg_team_role_fantasy_score,
      COUNT(*) AS maps
    FROM fantasy_team_map_role_analysis
    WHERE team_name IN ({top_16_team_names_sql})
      AND opponent_name IN ({top_16_team_names_sql})
    GROUP BY team_name
    ORDER BY avg_team_role_fantasy_score DESC
    """,
)


In [ ]:
show_query(
    "Игроки, где имя или позиция исправлены вручную",
    """
    SELECT
      team_name,
      raw_player_name,
      official_player_name,
      position_used,
      official_role,
      position_source
    FROM player_official_profiles_temp
    WHERE position_source = 'manual_account_id_override'
    ORDER BY team_name, position_used
    """,
    limit=100,
)


In [ ]:
show_query(
    "Игроки, где позиция восстановлена эвристически",
    """
    SELECT
      team_name,
      raw_player_name,
      official_player_name,
      position_used,
      official_role,
      avg_last_hits,
      avg_gpm,
      position_source
    FROM player_official_profiles_temp
    WHERE position_source = 'inferred_by_team_avg_last_hits_rank'
    ORDER BY team_name, position_used
    """,
    limit=100,
)


In [ ]:
print("Аналитический блок завершён. Для основных отчётов используйте:")
print("- player_game_fantasy_analysis: игрок на карту с официальным именем и позицией")
print("- fantasy_player_role_totals: итоги игрока по независимым картам")
print("- fantasy_team_map_role_analysis: avg core + mid + avg support по каждой карте")


<!-- EWC2026_V2_LIQUIPEDIA_ROLE_CATEGORY_LAYER -->
## V2: Liquipedia roster + role-category слой

Эта секция использует новую компактную базу `ewc_2026_fantasy_compact.sqlite`.

Что она добавляет/проверяет:

- `liquipedia_team_rosters`: составы и позиции `1..5` с главной страницы Liquipedia EWC 2026.
- `player_identity_registry`: единый реестр `account_id -> official_name + official_position`.
- `player_map_role_category_stats`: live-view для каждой команды на каждой карте `core_avg`, `mid`, `support_avg`.
- `team_map_role_category_summary`: компактная строка на команду-карту: средние коры + мидер + средние саппорты.

Если эти таблицы уже есть в базе, секция просто показывает их состояние. Если нужно полностью пересоздать слой в Colab, сначала запусти ячейки сбора данных выше, затем эту секцию.


In [ ]:
# EWC2026_V2_LIQUIPEDIA_ROLE_CATEGORY_LAYER
from __future__ import annotations

import json
import re
import sqlite3
import urllib.request
from collections import defaultdict
from pathlib import Path
from typing import Any

import pandas as pd

LIQUIPEDIA_EWC2026_URL = "https://liquipedia.net/dota2/Esports_World_Cup/2026"
LIQUIPEDIA_RAW_URL = LIQUIPEDIA_EWC2026_URL + "?action=raw"

LIQUIPEDIA_TEAM_TO_DB = {
    "1w 2026": "1w",
    "PARIVISION": "PVISION",
    "BetBoom Team": "BoomBoys",
    "PlayTime": "PTime",
    "Level UP": "Level UP esports",
    "Poor Rangers": "_PowerRangers",
    "L1 TEAM 2026": "L1 TEAM",
    "Inner Circle": "Inner Circle x Insanity",
}

ROLE_LABELS = {
    1: "pos1 carry/core",
    2: "pos2 mid",
    3: "pos3 offlane/core",
    4: "pos4 support",
    5: "pos5 support",
}


def _norm_name(value: str | None) -> str:
    value = (value or "").strip().lower().replace("`", "'")
    return re.sub(r"\s+", " ", value)


def _compact_name(value: str | None) -> str:
    return re.sub(r"[^a-z0-9]+", "", _norm_name(value))


def _name_keys(value: str | None) -> set[str]:
    keys = {_norm_name(value), _compact_name(value)}
    keys.discard("")
    return keys


def role_group(position: int) -> str:
    if position in (1, 3):
        return "core"
    if position == 2:
        return "mid"
    return "support"


def table_exists(connection: sqlite3.Connection, name: str) -> bool:
    row = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE name = ? AND type IN ('table', 'view')",
        (name,),
    ).fetchone()
    return row is not None


def _extract_team_participants_block(raw_text: str) -> str:
    start = raw_text.index("{{TeamParticipants")
    depth = 0
    i = start
    while i < len(raw_text):
        if raw_text.startswith("{{", i):
            depth += 1
            i += 2
            continue
        if raw_text.startswith("}}", i):
            depth -= 1
            i += 2
            if depth == 0:
                return raw_text[start:i]
            continue
        i += 1
    raise ValueError("TeamParticipants block is not closed")


def _split_top_level_opponents(block: str) -> list[str]:
    opponents: list[str] = []
    cursor = 0
    while True:
        marker = block.find("|{{Opponent|", cursor)
        if marker == -1:
            break
        pos = marker + 1
        depth = 0
        i = pos
        while i < len(block):
            if block.startswith("{{", i):
                depth += 1
                i += 2
                continue
            if block.startswith("}}", i):
                depth -= 1
                i += 2
                if depth == 0:
                    opponents.append(block[pos:i])
                    cursor = i
                    break
                continue
            i += 1
    return opponents


def _parse_person_params(person_template: str) -> dict[str, str]:
    inner = person_template.removeprefix("{{Person|").removesuffix("}}")
    params: dict[str, str] = {}
    positional: list[str] = []
    for part in inner.split("|"):
        if "=" in part:
            key, value = part.split("=", 1)
            params[key.strip()] = value.strip()
        elif part.strip():
            positional.append(part.strip())
    if positional:
        params["name"] = positional[0]
    return params


def fetch_liquipedia_roster_wikitext(cache_path: Path | None = None, force: bool = False) -> str:
    """Возвращает raw-wikitext страницы Liquipedia. Использует cache_path, если он есть."""
    cache_path = cache_path or (DB_PATH.parent / "liquipedia_ewc2026_raw.wiki")
    if cache_path.exists() and not force:
        return cache_path.read_text(encoding="utf-8")
    req = urllib.request.Request(
        LIQUIPEDIA_RAW_URL,
        headers={
            "User-Agent": "EWC2026DotaNotebook/2.0 (local educational analysis)",
            "Accept": "text/plain, */*;q=0.8",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        raw = response.read().decode(response.headers.get_content_charset() or "utf-8", errors="replace")
    cache_path.write_text(raw, encoding="utf-8")
    return raw


def parse_liquipedia_rosters_from_wikitext(raw_text: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    block = _extract_team_participants_block(raw_text)
    for opponent in _split_top_level_opponents(block):
        team = opponent.splitlines()[0].removeprefix("{{Opponent|").strip()
        db_team = LIQUIPEDIA_TEAM_TO_DB.get(team, team)
        for match in re.finditer(r"\{\{Person\|[^{}]*\}\}", opponent):
            params = _parse_person_params(match.group(0))
            role_value = params.get("role")
            if role_value not in {"1", "2", "3", "4", "5"}:
                continue
            position = int(role_value)
            rows.append(
                {
                    "db_team": db_team,
                    "liquipedia_team": team,
                    "official_position": position,
                    "official_name": params.get("name"),
                    "role_label": ROLE_LABELS[position],
                    "role_group": role_group(position),
                    "liquipedia_link": params.get("link"),
                    "liquipedia_id": params.get("id"),
                    "source_url": LIQUIPEDIA_EWC2026_URL,
                }
            )
    return pd.DataFrame(rows)


LIQUIPEDIA_NAME_ALIASES = {
    "atf": {"ammar_the_f", "ammar"},
    "no[o]ne-": {"noone", "no[o]ne", "noone-"},
    "noticed": {"dm"},
    "lari": {"larl"},
    "ainkrad": {"darknia"},
    "m1cke": {"micke", "miCKe"},
    "ace": {"saberlight", "saberlight-"},
    "fy": {"xinq"},
    "not me": {"miposhka"},
    "vazya": {"corrupted"},
    "zayac": {"kg_zayac"},
    "respect": {"mirage"},
    "ws": {"ws`"},
    "kj": {"kingjungles"},
    "darklord^": {"darklord,,`"},
    "4nalog": {"4nalog丶01"},
    "jabz": {"j"},
    "yamsun": {"surf's up"},
    "watson": {"医者watson`"},
    "malady": {"maladych"},
    "yopaj": {"yopaj-"},
    "darkmago": {"darkmago♥"},
    "frank": {"fr△nk"},
    "ghost": {"鬼"},
    "xm": {"^"},
    "y`": {"宇宙にきらめく エメラルド"},
    "shiro": {"imailisa"},
    "nicky`cool": {"nickycool"},
    "ta2000": {"naive-"},
    "hduo": {"till the end"},
}


def build_account_name_index(connection: sqlite3.Connection) -> dict[str, dict[int, set[str]]]:
    names: dict[str, dict[int, set[str]]] = defaultdict(lambda: defaultdict(set))
    if table_exists(connection, "player_game_fantasy_summary"):
        query = """
        SELECT DISTINCT f.team_name, f.account_id, f.player_name,
               c.opendota_name, c.opendota_personaname
        FROM player_game_fantasy_summary f
        LEFT JOIN opendota_player_identity_cache c
          ON c.match_id = f.match_id
         AND c.account_id = f.account_id
        """
        for team, account_id, player_name, od_name, od_persona in connection.execute(query):
            if account_id is None:
                continue
            for value in [player_name, od_name, od_persona]:
                if value:
                    names[team][int(account_id)].add(str(value))
    if table_exists(connection, "player_identity_registry"):
        for team, account_id, official, db_name, persona in connection.execute(
            "SELECT team_name, account_id, official_name, db_player_name, public_personaname FROM player_identity_registry"
        ):
            for value in [official, db_name, persona]:
                if value:
                    names[team][int(account_id)].add(str(value))
    return names


def attach_accounts_to_liquipedia_roster(connection: sqlite3.Connection, roster_df: pd.DataFrame) -> pd.DataFrame:
    account_names = build_account_name_index(connection)
    out_rows: list[dict[str, Any]] = []
    unresolved: list[dict[str, Any]] = []
    for team, team_df in roster_df.groupby("db_team", sort=True):
        used: set[int] = set()
        accounts = account_names.get(team, {})
        for row in team_df.sort_values("official_position").to_dict("records"):
            official_name = row["official_name"]
            lp_keys = _name_keys(official_name)
            alias_keys = set()
            for alias in LIQUIPEDIA_NAME_ALIASES.get(_norm_name(official_name), set()):
                alias_keys.update(_name_keys(alias))
            candidates = []
            for account_id, known_names in accounts.items():
                if account_id in used:
                    continue
                known_keys = set()
                for value in known_names:
                    known_keys.update(_name_keys(value))
                if (lp_keys | alias_keys) & known_keys:
                    score = 100 if lp_keys & known_keys else 80
                    candidates.append((score, account_id, ", ".join(sorted(known_names))))
            candidates.sort(reverse=True)
            if not candidates:
                unresolved.append(row)
                continue
            _, account_id, matched_names = candidates[0]
            used.add(account_id)
            row["account_id"] = account_id
            row["matched_names"] = matched_names
            out_rows.append(row)
    if unresolved:
        raise RuntimeError(f"Не удалось сопоставить {len(unresolved)} игроков Liquipedia с account_id: {unresolved[:5]}")
    result = pd.DataFrame(out_rows)
    if len(result) != len(roster_df):
        raise RuntimeError(f"Ожидалось {len(roster_df)} roster rows, получено {len(result)}")
    return result


def upsert_liquipedia_identity_layer(connection: sqlite3.Connection, roster_with_accounts: pd.DataFrame) -> None:
    connection.executescript(
        """
        CREATE TABLE IF NOT EXISTS liquipedia_team_rosters (
            db_team TEXT NOT NULL,
            liquipedia_team TEXT NOT NULL,
            account_id INTEGER NOT NULL,
            official_name TEXT NOT NULL,
            official_position INTEGER NOT NULL,
            role_label TEXT NOT NULL,
            role_group TEXT NOT NULL,
            liquipedia_link TEXT,
            liquipedia_id TEXT,
            matched_names TEXT,
            source_url TEXT NOT NULL,
            fetched_at_utc TEXT NOT NULL DEFAULT (datetime('now')),
            PRIMARY KEY (db_team, account_id)
        );

        CREATE TABLE IF NOT EXISTS player_identity_registry (
            account_id INTEGER NOT NULL,
            team_name TEXT NOT NULL,
            official_name TEXT NOT NULL,
            db_player_name TEXT,
            public_personaname TEXT,
            official_position INTEGER NOT NULL,
            role_label TEXT NOT NULL,
            role_group TEXT NOT NULL,
            position_source TEXT NOT NULL,
            identity_source TEXT NOT NULL,
            confidence_score REAL NOT NULL,
            confidence_label TEXT NOT NULL,
            maps_seen INTEGER NOT NULL,
            maps_at_position INTEGER NOT NULL,
            avg_fantasy_score REAL,
            best_map_fantasy_score REAL,
            source_name TEXT NOT NULL,
            source_url TEXT,
            resolved_at_utc TEXT NOT NULL DEFAULT (datetime('now')),
            notes TEXT,
            PRIMARY KEY (account_id, team_name)
        );

        CREATE TABLE IF NOT EXISTS player_identity_sources (
            account_id INTEGER NOT NULL,
            team_name TEXT NOT NULL,
            source_name TEXT NOT NULL,
            source_url TEXT,
            source_type TEXT NOT NULL,
            confidence_score REAL NOT NULL,
            extracted_name TEXT,
            extracted_position INTEGER,
            fetched_at_utc TEXT,
            notes TEXT,
            PRIMARY KEY (account_id, team_name, source_name, source_type)
        );
        DELETE FROM liquipedia_team_rosters;
        """
    )
    for row in roster_with_accounts.to_dict("records"):
        account_id = int(row["account_id"])
        team = row["db_team"]
        stats = connection.execute(
            """
            SELECT COUNT(*), ROUND(AVG(player_map_fantasy_score), 2), ROUND(MAX(player_map_fantasy_score), 2)
            FROM player_game_fantasy_summary
            WHERE team_name = ? AND account_id = ?
            """,
            (team, account_id),
        ).fetchone()
        maps_seen, avg_score, best_score = stats if stats else (0, None, None)
        existing = connection.execute(
            """
            SELECT db_player_name, public_personaname
            FROM player_identity_registry
            WHERE team_name = ? AND account_id = ?
            """,
            (team, account_id),
        ).fetchone()
        db_player_name, public_personaname = existing if existing else (None, None)
        values = (
            team,
            row["liquipedia_team"],
            account_id,
            row["official_name"],
            int(row["official_position"]),
            row["role_label"],
            row["role_group"],
            row.get("liquipedia_link"),
            row.get("liquipedia_id"),
            row.get("matched_names"),
            LIQUIPEDIA_EWC2026_URL,
        )
        connection.execute(
            """
            INSERT OR REPLACE INTO liquipedia_team_rosters(
                db_team, liquipedia_team, account_id, official_name, official_position,
                role_label, role_group, liquipedia_link, liquipedia_id, matched_names, source_url
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            values,
        )
        connection.execute(
            """
            INSERT INTO player_identity_registry(
                account_id, team_name, official_name, db_player_name, public_personaname,
                official_position, role_label, role_group, position_source, identity_source,
                confidence_score, confidence_label, maps_seen, maps_at_position,
                avg_fantasy_score, best_map_fantasy_score, source_name, source_url,
                resolved_at_utc, notes
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, datetime('now'), ?)
            ON CONFLICT(account_id, team_name) DO UPDATE SET
                official_name = excluded.official_name,
                official_position = excluded.official_position,
                role_label = excluded.role_label,
                role_group = excluded.role_group,
                position_source = excluded.position_source,
                identity_source = excluded.identity_source,
                confidence_score = excluded.confidence_score,
                confidence_label = excluded.confidence_label,
                maps_seen = excluded.maps_seen,
                maps_at_position = excluded.maps_at_position,
                avg_fantasy_score = excluded.avg_fantasy_score,
                best_map_fantasy_score = excluded.best_map_fantasy_score,
                source_name = excluded.source_name,
                source_url = excluded.source_url,
                resolved_at_utc = excluded.resolved_at_utc,
                notes = excluded.notes
            """,
            (
                account_id,
                team,
                row["official_name"],
                db_player_name,
                public_personaname,
                int(row["official_position"]),
                row["role_label"],
                row["role_group"],
                "liquipedia_participants_roster",
                "liquipedia_participants_roster",
                0.98,
                "high_liquipedia_roster_match",
                int(maps_seen or 0),
                int(maps_seen or 0),
                avg_score,
                best_score,
                "Liquipedia EWC 2026 participants",
                LIQUIPEDIA_EWC2026_URL,
                f"Liquipedia team={row['liquipedia_team']}; matched_names={row.get('matched_names')}",
            ),
        )
        connection.execute(
            """
            INSERT OR REPLACE INTO player_identity_sources(
                account_id, team_name, source_name, source_url, source_type,
                confidence_score, extracted_name, extracted_position, fetched_at_utc, notes
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, datetime('now'), ?)
            """,
            (
                account_id,
                team,
                "Liquipedia EWC 2026 participants",
                LIQUIPEDIA_EWC2026_URL,
                "participant_roster_wikitext",
                0.98,
                row["official_name"],
                int(row["official_position"]),
                f"matched_names={row.get('matched_names')}",
            ),
        )
    connection.execute(
        "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
        ("liquipedia_team_rosters_rows", str(len(roster_with_accounts))),
    )
    connection.execute(
        "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
        ("player_identity_registry_version", "identity_registry_v2_liquipedia_ewc2026_rosters"),
    )
    connection.execute(
        "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
        ("player_identity_registry_source_url", LIQUIPEDIA_EWC2026_URL),
    )


def rebuild_role_category_tables(connection: sqlite3.Connection) -> None:
    """Пересоздает агрегаты: avg pos1+3, pos2, avg pos4+5 для каждой команды на каждой карте."""
    connection.executescript(
        """
        DROP VIEW IF EXISTS team_map_role_category_summary;
        DROP VIEW IF EXISTS player_map_role_category_stats;
        DROP TABLE IF EXISTS player_map_role_category_stats;

        CREATE VIEW player_map_role_category_stats AS
        WITH base AS (
            SELECT
                f.*,
                pir.official_name,
                pir.official_position,
                pir.role_label,
                pir.role_group,
                pir.confidence_score,
                pir.confidence_label,
                CASE
                    WHEN pir.official_position IN (1, 3) THEN 'core_avg'
                    WHEN pir.official_position = 2 THEN 'mid'
                    WHEN pir.official_position IN (4, 5) THEN 'support_avg'
                END AS role_category,
                CASE
                    WHEN pir.official_position IN (1, 3) THEN 'Average of official pos1 and pos3'
                    WHEN pir.official_position = 2 THEN 'Official pos2 mid'
                    WHEN pir.official_position IN (4, 5) THEN 'Average of official pos4 and pos5'
                END AS role_category_label
            FROM player_game_fantasy_summary f
            JOIN player_identity_registry pir
              ON pir.account_id = f.account_id
             AND pir.team_name = f.team_name
            WHERE pir.official_position BETWEEN 1 AND 5
        )
        SELECT
            match_id,
            match_date,
            series_id,
            league_id,
            team_name,
            opponent_name,
            MAX(side) AS side,
            MAX(won) AS won,
            MAX(duration_sec) AS duration_sec,
            role_category,
            MAX(role_category_label) AS role_category_label,
            GROUP_CONCAT(DISTINCT CAST(official_position AS TEXT)) AS included_positions,
            COUNT(*) AS players_count,
            GROUP_CONCAT(official_name, ', ') AS player_names,
            GROUP_CONCAT(CAST(account_id AS TEXT), ', ') AS account_ids,
            GROUP_CONCAT(COALESCE(hero_name, ''), ', ') AS hero_names,
            ROUND(AVG(kills), 2) AS kills,
            ROUND(AVG(deaths), 2) AS deaths,
            ROUND(AVG(assists), 2) AS assists,
            ROUND(AVG(last_hits), 2) AS last_hits,
            ROUND(AVG(denies), 2) AS denies,
            ROUND(AVG(creep_score), 2) AS creep_score,
            ROUND(AVG(gpm), 2) AS gpm,
            ROUND(AVG(xpm), 2) AS xpm,
            ROUND(AVG(observer_wards_placed), 2) AS observer_wards_placed,
            ROUND(AVG(camps_stacked), 2) AS camps_stacked,
            ROUND(AVG(runes_grabbed), 2) AS runes_grabbed,
            ROUND(AVG(watchers_taken), 2) AS watchers_taken,
            ROUND(AVG(lotus_units), 2) AS lotus_units,
            ROUND(AVG(roshan_kills), 2) AS roshan_kills,
            ROUND(AVG(tormentor_kills), 2) AS tormentor_kills,
            ROUND(AVG(courier_kills), 2) AS courier_kills,
            ROUND(AVG(first_blood), 2) AS first_blood,
            ROUND(AVG(stuns_sec), 2) AS stuns_sec,
            ROUND(AVG(smokes_used), 2) AS smokes_used,
            ROUND(AVG(team_kills), 2) AS team_kills,
            ROUND(AVG(teamfight_participation_ratio), 4) AS teamfight_participation_ratio,
            ROUND(AVG(player_map_fantasy_score), 2) AS role_category_fantasy_score,
            ROUND(SUM(player_map_fantasy_score), 2) AS stored_player_scores_sum,
            ROUND(AVG(player_map_fantasy_score), 2) AS stored_player_scores_avg,
            ROUND(MIN(confidence_score), 2) AS min_identity_confidence,
            GROUP_CONCAT(DISTINCT confidence_label) AS confidence_labels,
            CASE
                WHEN role_category = 'mid' THEN 'single_official_pos2'
                WHEN role_category = 'core_avg' THEN 'average_official_pos1_pos3'
                WHEN role_category = 'support_avg' THEN 'average_official_pos4_pos5'
            END AS aggregation_method,
            'official_position_complete_team_map' AS position_resolution_method,
            MAX(scoring_version) AS scoring_version,
            MAX(source_dotabuff_url) AS source_dotabuff_url,
            CASE
                WHEN COUNT(*) = 1 AND role_category = 'mid' THEN 'complete_official_positions'
                WHEN COUNT(*) = 2 AND role_category IN ('core_avg', 'support_avg') THEN 'complete_official_positions'
                ELSE 'incomplete_role_category_players'
            END AS data_quality_note
        FROM base
        GROUP BY match_id, match_date, series_id, league_id, team_name, opponent_name, role_category;

        CREATE VIEW team_map_role_category_summary AS
        SELECT
            match_id,
            match_date,
            team_name,
            opponent_name,
            MAX(CASE WHEN role_category = 'core_avg' THEN role_category_fantasy_score END) AS avg_core_fantasy_score,
            MAX(CASE WHEN role_category = 'mid' THEN role_category_fantasy_score END) AS mid_fantasy_score,
            MAX(CASE WHEN role_category = 'support_avg' THEN role_category_fantasy_score END) AS avg_support_fantasy_score,
            ROUND(
                COALESCE(MAX(CASE WHEN role_category = 'core_avg' THEN role_category_fantasy_score END), 0)
              + COALESCE(MAX(CASE WHEN role_category = 'mid' THEN role_category_fantasy_score END), 0)
              + COALESCE(MAX(CASE WHEN role_category = 'support_avg' THEN role_category_fantasy_score END), 0),
                2
            ) AS team_role_fantasy_score,
            MAX(CASE WHEN role_category = 'core_avg' THEN player_names END) AS core_players,
            MAX(CASE WHEN role_category = 'mid' THEN player_names END) AS mid_player,
            MAX(CASE WHEN role_category = 'support_avg' THEN player_names END) AS support_players,
            MIN(min_identity_confidence) AS min_identity_confidence,
            GROUP_CONCAT(DISTINCT confidence_labels) AS confidence_labels,
            GROUP_CONCAT(DISTINCT position_resolution_method) AS position_resolution_methods,
            GROUP_CONCAT(DISTINCT data_quality_note) AS data_quality_notes
        FROM player_map_role_category_stats
        GROUP BY match_id, match_date, team_name, opponent_name;
        """
    )
    count = connection.execute("SELECT COUNT(*) FROM player_map_role_category_stats").fetchone()[0]
    connection.execute(
        "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
        ("player_map_role_category_stats_rows", str(count)),
    )
    connection.execute(
        "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
        ("role_category_stats_version", "role_category_map_stats_v1_avg_core_mid_avg_support"),
    )


def sync_liquipedia_and_role_category_layer(connection: sqlite3.Connection = conn, fetch_if_missing: bool = False) -> dict[str, Any]:
    """Главная функция секции: проверяет/создает roster layer и пересобирает role-category агрегаты."""
    result: dict[str, Any] = {}
    if table_exists(connection, "liquipedia_team_rosters"):
        result["liquipedia_team_rosters_before"] = pd.read_sql_query(
            "SELECT COUNT(*) AS rows, COUNT(DISTINCT db_team) AS teams FROM liquipedia_team_rosters",
            connection,
        ).to_dict("records")[0]
    elif fetch_if_missing:
        raw = fetch_liquipedia_roster_wikitext()
        roster = parse_liquipedia_rosters_from_wikitext(raw)
        roster_accounts = attach_accounts_to_liquipedia_roster(connection, roster)
        upsert_liquipedia_identity_layer(connection, roster_accounts)
        result["liquipedia_team_rosters_created"] = len(roster_accounts)
    else:
        result["liquipedia_team_rosters_before"] = "missing; call sync_liquipedia_and_role_category_layer(fetch_if_missing=True)"

    if table_exists(connection, "player_identity_registry"):
        rebuild_role_category_tables(connection)
        connection.commit()
        result["player_identity_registry_rows"] = connection.execute("SELECT COUNT(*) FROM player_identity_registry").fetchone()[0]
        result["role_category_rows"] = connection.execute("SELECT COUNT(*) FROM player_map_role_category_stats").fetchone()[0]
        result["team_role_summary_rows"] = connection.execute("SELECT COUNT(*) FROM team_map_role_category_summary").fetchone()[0]
        result["position_resolution"] = pd.read_sql_query(
            """
            SELECT position_resolution_method, COUNT(*) AS rows
            FROM player_map_role_category_stats
            GROUP BY position_resolution_method
            """,
            connection,
        ).to_dict("records")
    return result


def show_official_roster(team: str | None = None) -> pd.DataFrame:
    where = ""
    params: list[Any] = []
    if team:
        where = "WHERE lower(team_name) = lower(?) OR lower(team_name) IN (SELECT lower(canonical_team_name) FROM team_aliases WHERE lower(alias)=lower(?))"
        params = [team, team]
    return pd.read_sql_query(
        f"""
        SELECT team_name, official_position, official_name, account_id,
               role_label, confidence_label, source_url
        FROM player_identity_registry
        {where}
        ORDER BY team_name, official_position
        """,
        conn,
        params=params,
    )


def show_team_role_fantasy(team: str | None = None, limit: int = 20) -> pd.DataFrame:
    where = ""
    params: list[Any] = []
    if team:
        where = "WHERE lower(team_name) = lower(?) OR lower(team_name) IN (SELECT lower(canonical_team_name) FROM team_aliases WHERE lower(alias)=lower(?))"
        params = [team, team]
    return pd.read_sql_query(
        f"""
        SELECT match_id, match_date, team_name, opponent_name,
               avg_core_fantasy_score, mid_fantasy_score, avg_support_fantasy_score,
               team_role_fantasy_score, core_players, mid_player, support_players
        FROM team_map_role_category_summary
        {where}
        ORDER BY team_role_fantasy_score DESC
        LIMIT {int(limit)}
        """,
        conn,
        params=params,
    )


sync_status = sync_liquipedia_and_role_category_layer(conn, fetch_if_missing=False)
print(json.dumps(sync_status, ensure_ascii=False, indent=2))
display(show_official_roster("BetBoom Team"))
display(show_team_role_fantasy("BetBoom Team", limit=10))


<!-- EWC2026_V3_FANTASY_PROFILE_LAYER -->
## V3: единый fantasy profile слой

После миграции БД fantasy-очки считаются не как единственное число, а как профиль:

- `my_current_banner_official_roles`: текущие личные коэффициенты, пересчитанные по официальным позициям Liquipedia.
- `legacy_stored_banner_score`: historical label; в актуальной базе `player_map_fantasy_score` уже синхронизирован с current default profile.

Главные таблицы:

- `fantasy_scoring_profiles`
- `fantasy_scoring_profile_stats`
- `fantasy_player_map_stat_points`
- `fantasy_player_map_scores`
- `fantasy_team_role_map_scores`
- `fantasy_pick_value`

Главные views:

- `v_fantasy_default_player_map_scores`
- `v_fantasy_default_pick_value`
- `v_fantasy_default_team_role_map_summary`


In [ ]:
# EWC2026_V3_FANTASY_PROFILE_LAYER
def get_default_fantasy_profile(connection: sqlite3.Connection = conn) -> str:
    row = connection.execute(
        "SELECT profile_id FROM fantasy_scoring_profiles WHERE is_default = 1 LIMIT 1"
    ).fetchone()
    return row[0] if row else "my_current_banner_official_roles"


def show_fantasy_profile_status(connection: sqlite3.Connection = conn) -> pd.DataFrame:
    rows = []
    for name in [
        "fantasy_scoring_profiles",
        "fantasy_scoring_stat_catalog",
        "fantasy_scoring_profile_stats",
        "fantasy_scoring_profile_banners",
        "fantasy_player_map_stat_points",
        "fantasy_player_map_scores",
        "fantasy_team_role_map_scores",
        "fantasy_pick_value",
        "v_fantasy_default_pick_value",
        "v_fantasy_default_team_role_map_summary",
    ]:
        exists = connection.execute(
            "SELECT 1 FROM sqlite_master WHERE name = ? AND type IN ('table', 'view')",
            (name,),
        ).fetchone() is not None
        count = connection.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] if exists else None
        rows.append({"object": name, "exists": exists, "rows": count})
    return pd.DataFrame(rows)


def show_top_pick_value(
    role_group: str | None = None,
    position: int | None = None,
    profile_id: str | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    profile_id = profile_id or get_default_fantasy_profile(connection)
    clauses = ["profile_id = ?"]
    params: list[object] = [profile_id]
    if role_group:
        clauses.append("role_group = ?")
        params.append(role_group)
    if position:
        clauses.append("official_position = ?")
        params.append(position)
    where = "WHERE " + " AND ".join(clauses)
    return pd.read_sql_query(
        f"""
        SELECT profile_id, official_name, team_name, official_position, role_group,
               maps_seen, avg_score, best_score, floor_score,
               avg_abs_deviation, consistency_score, ceiling_score, pick_value_score
        FROM fantasy_pick_value
        {where}
        ORDER BY pick_value_score DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


def show_profile_role_map_summary(
    team: str | None = None,
    profile_id: str | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    profile_id = profile_id or get_default_fantasy_profile(connection)
    clauses = ["profile_id = ?"]
    params: list[object] = [profile_id]
    if team:
        resolved = canonical_team_name(team)
        clauses.append("team_name = ?")
        params.append(resolved)
    where = "WHERE " + " AND ".join(clauses)
    return pd.read_sql_query(
        f"""
        SELECT profile_id, match_id, match_date, team_name, opponent_name,
               avg_core_fantasy_score, mid_fantasy_score, avg_support_fantasy_score,
               team_role_fantasy_score, core_players, mid_player, support_players
        FROM v_fantasy_team_role_map_summary_by_profile
        {where}
        ORDER BY team_role_fantasy_score DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


display(show_fantasy_profile_status())
print("Default fantasy profile:", get_default_fantasy_profile())
display(show_top_pick_value(position=1, limit=15))
display(show_profile_role_map_summary(team="BetBoom Team", limit=10))


<!-- EWC2026_V4_MATCH_STAGE_LAYER -->
## V4: stage-классификация игр

Добавлены поля стадии матча:

- `stage_name`: точная стадия (`Group Stage`, `Survival Stage`, `Playoffs`).
- `stage_bucket`: аналитический bucket (`group_stage` или `playoffs`).
- `is_group_stage_bucket`: `1` для Group Stage + Survival.
- `is_main_playoff`: `1` только для основного Playoffs.

По правилу проекта `Survival Stage` считается частью `group_stage`, потому что это не основной playoff.


In [ ]:
# EWC2026_V4_MATCH_STAGE_LAYER
def show_match_stage_status(connection: sqlite3.Connection = conn) -> pd.DataFrame:
    return pd.read_sql_query(
        """
        SELECT match_date, stage_name, stage_bucket,
               COUNT(*) AS maps
        FROM match_stage_registry
        GROUP BY match_date, stage_name, stage_bucket
        ORDER BY match_date
        """,
        connection,
    )


def show_matches_by_stage(stage_bucket: str | None = None, limit: int = 50, connection: sqlite3.Connection = conn) -> pd.DataFrame:
    where = ""
    params: list[object] = []
    if stage_bucket:
        where = "WHERE stage_bucket = ?"
        params.append(stage_bucket)
    return pd.read_sql_query(
        f"""
        SELECT match_id, match_date, stage_name, stage_bucket,
               winner_name, loser_name, radiant_name, dire_name,
               radiant_score, dire_score, source_dotabuff_url
        FROM match_evidence
        {where}
        ORDER BY match_date, match_id
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


def show_profile_role_map_summary(
    team: str | None = None,
    profile_id: str | None = None,
    stage_bucket: str | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    profile_id = profile_id or get_default_fantasy_profile(connection)
    clauses = ["profile_id = ?"]
    params: list[object] = [profile_id]
    if team:
        resolved = canonical_team_name(team)
        clauses.append("team_name = ?")
        params.append(resolved)
    if stage_bucket:
        clauses.append("stage_bucket = ?")
        params.append(stage_bucket)
    where = "WHERE " + " AND ".join(clauses)
    return pd.read_sql_query(
        f"""
        SELECT profile_id, match_id, match_date, stage_name, stage_bucket,
               team_name, opponent_name, avg_core_fantasy_score,
               mid_fantasy_score, avg_support_fantasy_score,
               team_role_fantasy_score, core_players, mid_player, support_players
        FROM v_fantasy_team_role_map_summary_by_profile
        {where}
        ORDER BY team_role_fantasy_score DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


display(show_match_stage_status())
display(show_matches_by_stage("playoffs", limit=10))
display(show_profile_role_map_summary(team="BetBoom Team", stage_bucket="playoffs", limit=10))


<!-- EWC2026_V5_RELIABILITY_MODEL -->
## V5: модель надежности fantasy-пика

Модель оценивает не среднюю карту, а **повторяемый потолок** под механику fantasy:

1. Для каждой серии считается `best2_series_score`: сумма двух лучших карт игрока в этой серии.
2. Train: `stage_bucket='group_stage'`, то есть Group Stage + Survival.
3. Test/backtest: `stage_bucket='playoffs'`.
4. Итоговая оценка `reliability_score_1_100` — percentile rank 1–100 внутри роли/слота.
5. Один выброс штрафуется через `spike_gap = best2_series_score - second_best2_series_score`.

Главные views:

- `v_fantasy_reliable_players_top`
- `v_fantasy_reliable_role_slots_top`
- `v_fantasy_reliability_backtest_default`


In [ ]:
# EWC2026_V5_RELIABILITY_MODEL
def show_reliability_status(connection: sqlite3.Connection = conn) -> pd.DataFrame:
    rows = []
    for name in [
        "fantasy_player_series_scores",
        "fantasy_role_slot_series_scores",
        "fantasy_reliability_player_predictions",
        "fantasy_reliability_role_slot_predictions",
        "fantasy_reliability_temporal_backtest_predictions",
        "fantasy_reliability_model_evaluation",
        "v_fantasy_reliable_players_top",
        "v_fantasy_reliable_role_slots_top",
    ]:
        exists = connection.execute(
            "SELECT 1 FROM sqlite_master WHERE name = ? AND type IN ('table', 'view')",
            (name,),
        ).fetchone() is not None
        count = connection.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] if exists else None
        rows.append({"object": name, "exists": exists, "rows": count})
    return pd.DataFrame(rows)


def show_reliable_players(
    role_group: str | None = None,
    position: int | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    clauses = []
    params: list[object] = []
    if role_group:
        clauses.append("role_group = ?")
        params.append(role_group)
    if position:
        clauses.append("official_position = ?")
        params.append(position)
    where = "WHERE " + " AND ".join(clauses) if clauses else ""
    return pd.read_sql_query(
        f"""
        SELECT reliability_score_1_100, ceiling_reliability_1_100,
               official_name, team_name, official_position, role_group,
               predicted_score_raw, train_best2_series_score,
               train_second_best2_series_score, spike_gap,
               repeatability_ratio, train_series_seen,
               actual_test_best2_series_score
        FROM v_fantasy_reliable_players_top
        {where}
        ORDER BY reliability_score_1_100 DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


def show_reliable_role_slots(
    role_slot: str | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    clauses = []
    params: list[object] = []
    if role_slot:
        clauses.append("role_slot = ?")
        params.append(role_slot)
    where = "WHERE " + " AND ".join(clauses) if clauses else ""
    return pd.read_sql_query(
        f"""
        SELECT reliability_score_1_100, ceiling_reliability_1_100,
               team_name, role_slot, player_names,
               predicted_score_raw, train_best2_series_score,
               train_second_best2_series_score, spike_gap,
               repeatability_ratio, train_series_seen,
               actual_test_best2_series_score
        FROM v_fantasy_reliable_role_slots_top
        {where}
        ORDER BY role_slot, reliability_score_1_100 DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


def show_reliability_backtest(connection: sqlite3.Connection = conn) -> pd.DataFrame:
    return pd.read_sql_query(
        """
        SELECT entity_type, segment_name, n_test, mae, rmse,
               spearman_corr, top5_overlap_rate, top10_overlap_rate
        FROM v_fantasy_reliability_backtest_default
        ORDER BY entity_type, segment_name
        """,
        connection,
    )


display(show_reliability_status())
display(show_reliable_players(position=1, limit=15))
display(show_reliable_role_slots(role_slot="core_pair", limit=15))
display(show_reliability_backtest())


<!-- EWC2026_V6_SUPPORT_QUALITY_SCOPE -->
## V6: качество support-статистики и scope рекомендаций

В базе support-статистика помечена как `support_low_stat_coverage`: она остается доступной для явного анализа, но не используется в рекомендациях по умолчанию.

Практическое правило:

- дефолтные fantasy-рекомендации: позиции 1-3, `core_pair`, `mid_single`;
- саппорты: только по явному запросу и с предупреждением о низкой надежности данных;
- сырые очки и статистика саппортов не удаляются.


In [ ]:
# EWC2026_V6_SUPPORT_QUALITY_SCOPE
def show_support_quality_status(connection: sqlite3.Connection = conn) -> pd.DataFrame:
    objects = [
        "fantasy_reliability_role_quality",
        "v_fantasy_reliable_recommended_players_top",
        "v_fantasy_reliable_recommended_role_slots_top",
        "v_fantasy_reliable_support_caveat",
    ]
    rows = []
    for name in objects:
        exists = connection.execute(
            "SELECT 1 FROM sqlite_master WHERE name = ? AND type IN ('table', 'view')",
            (name,),
        ).fetchone() is not None
        count = connection.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] if exists else None
        rows.append({"object": name, "exists": exists, "rows": count})
    return pd.DataFrame(rows)


def show_reliable_players_recommended(
    role_group: str | None = None,
    position: int | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    clauses = []
    params: list[object] = []
    if role_group:
        clauses.append("role_group = ?")
        params.append(role_group)
    if position:
        clauses.append("official_position = ?")
        params.append(int(position))
    where = "WHERE " + " AND ".join(clauses) if clauses else ""
    return pd.read_sql_query(
        f"""
        SELECT reliability_score_1_100, official_name, team_name,
               official_position, role_group, predicted_score_raw,
               train_best2_series_score, train_second_best2_series_score,
               spike_gap, repeatability_ratio, train_series_seen
        FROM v_fantasy_reliable_recommended_players_top
        {where}
        ORDER BY reliability_score_1_100 DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


def show_reliable_supports_low_confidence(
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    return pd.read_sql_query(
        f"""
        SELECT reliability_score_1_100, official_name, team_name,
               official_position, role_group, predicted_score_raw,
               train_best2_series_score, train_second_best2_series_score,
               spike_gap, repeatability_ratio, data_quality_label
        FROM v_fantasy_reliable_players_top
        WHERE role_group = 'support'
        ORDER BY reliability_score_1_100 DESC
        LIMIT {int(limit)}
        """,
        connection,
    )


def show_reliable_role_slots_recommended(
    role_slot: str | None = None,
    limit: int = 25,
    connection: sqlite3.Connection = conn,
) -> pd.DataFrame:
    clauses = []
    params: list[object] = []
    if role_slot:
        clauses.append("role_slot = ?")
        params.append(role_slot)
    where = "WHERE " + " AND ".join(clauses) if clauses else ""
    return pd.read_sql_query(
        f"""
        SELECT reliability_score_1_100, team_name, role_slot,
               player_names, predicted_score_raw, train_best2_series_score,
               train_second_best2_series_score, spike_gap,
               repeatability_ratio, train_series_seen
        FROM v_fantasy_reliable_recommended_role_slots_top
        {where}
        ORDER BY role_slot, reliability_score_1_100 DESC
        LIMIT {int(limit)}
        """,
        connection,
        params=params,
    )


display(show_support_quality_status())
display(pd.read_sql_query(
    """
    SELECT entity_type, role_key, data_quality_label, include_in_default_recommendations
    FROM fantasy_reliability_role_quality
    ORDER BY entity_type, role_key
    """,
    conn,
))
display(show_reliable_players_recommended(limit=10))
